# 공원 Polygon → WalkEdge 매핑 검증

서울시 생활권계획 시설(공원) Polygon을 도보 네트워크 WalkEdge에 연결할 때 다음 네 가지 방식을 비교합니다.

- Polygon과 조금이라도 닿음 (`ST_Intersects`)
- Edge 중점이 Polygon 내부에 있음
- Edge 길이의 10% 이상이 Polygon 내부에 있음
- Edge 길이의 50% 이상이 Polygon 내부에 있음

기존 도보 네트워크의 `공원,녹지` 플래그는 비교 기준으로 사용하지만, 외부 Polygon과 정의가 다를 수 있으므로 정답으로 간주하지 않습니다. DB에 접근하거나 데이터를 적재하지 않는 로컬 분석입니다.

In [4]:
import runpy
from pathlib import Path
from IPython.display import display

script_path = Path("analysis/raw/park_walkedge_mapping_validation.py")
if not script_path.exists():
    script_path = Path("park_walkedge_mapping_validation.py")

analysis = runpy.run_path(script_path)
summary, disagreements, proximity = analysis["run_analysis"]()
display(summary)
display(proximity)
display(disagreements.head(20))

,method,total_edges,selected_edges,selected_ratio,raw_park_green_edges,both_edges,method_only_edges,raw_only_edges,precision_vs_raw,recall_vs_raw,jaccard_vs_raw,park_polygons
0,intersects,279016,13194,0.047288,1162,125,13069,1037,0.009474,0.107573,0.008784,1888
1,midpoint_within,279016,10346,0.037080,1162,79,10267,1083,0.007636,0.067986,0.006912,1888
2,overlap_ratio_ge_0.1,279016,12166,0.043603,1162,115,12051,1047,0.009453,0.098967,0.008704,1888
3,overlap_ratio_ge_0.5,279016,10207,0.036582,1162,77,10130,1085,0.007544,0.066265,0.006819,1888


,distance_m,matched_raw_park_green_edges,total_raw_park_green_edges,matched_ratio
0,0,125,1162,0.107573
1,5,159,1162,0.136833
2,10,172,1162,0.148021
3,30,251,1162,0.216007
4,50,330,1162,0.283993
5,100,477,1162,0.410499


,link_id,district,neighborhood,source_length_m,raw_is_park_green,intersects_park,midpoint_within_park,park_overlap_ratio
14659,147197,중구,신당동,100.009,True,True,False,0.897113
37555,277008,성동구,하왕십리동,35.961,True,True,False,0.794105
37835,126150,성동구,금호동2가,75.692,True,True,False,0.483201
215630,50649,동작구,본동,20.965,True,True,False,0.478592
249219,227980,강남구,수서동,38.611,True,True,False,0.477888
195239,151506,영등포구,양화동,247.385,True,True,False,0.472571
19127,182960,용산구,용산동6가,60.205,True,True,False,0.468264
189069,213431,금천구,가산동,71.156,True,True,False,0.459797
22507,161826,용산구,이촌동,100.315,True,True,False,0.455416
76358,194573,성북구,삼선동1가,3.607,True,True,False,0.450249


## 2026-07-23 실행 결과

| 방식 | 선택 Edge | 전체 비율 | 원본 플래그와 동시 일치 |
| --- | ---: | ---: | ---: |
| 조금이라도 교차 | 13,194 | 4.73% | 125 |
| 중점 포함 | 10,346 | 3.71% | 79 |
| 내부 길이 10% 이상 | 12,166 | 4.36% | 115 |
| 내부 길이 50% 이상 | 10,207 | 3.66% | 77 |

전체 WalkEdge는 279,016개, 원본 `공원,녹지=1` Edge는 1,162개, 공원 Polygon은 1,888개입니다.

원본 플래그 Edge 중 외부 공원 Polygon과 직접 교차하는 비율은 10.76%이고, 50m 이내까지 넓혀도 28.40%, 100m 이내는 41.05%입니다. 따라서 낮은 일치율의 주원인은 교차 임계값이 아니라 두 자료가 표현하는 공원·녹지의 범위가 서로 다르기 때문입니다.

## 판정

1. 현재 Collector의 `ST_Intersects`만으로 `nature_score=1`을 덮어쓰면 안 됩니다. 경계에 살짝 닿는 Edge까지 포함하며, 원본 플래그와도 같은 의미가 아닙니다.
2. 원본 `공원,녹지` 플래그와 외부 Polygon 판정은 출처가 다른 별도 속성으로 보존해야 합니다.
3. Polygon 정보는 이진값으로 줄이지 않고, Edge 길이 중 공원 내부에 포함되는 비율인 `park_overlap_ratio`로 보존합니다.
4. 최종 자연 점수는 `raw_is_park_green`과 `park_overlap_ratio`를 사용할 수 있지만, 결합식은 데이터 적재와 분리해 Score 정책에서 확정해야 합니다.
5. 위 정책을 코드에 반영한 뒤에만 구형 DB를 초기화하고 최종 재적재합니다.